**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment2/'
FOLDERNAME = 'cs231n/assignments/assignment2/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载

# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# PyTorch 简介

在本次作业中，你已经写了大量代码，实现了许多神经网络功能。Dropout、Batch Norm 和二维卷积都是计算机视觉深度学习中的重要组件。你也花了很多精力让代码高效、向量化。

不过在本作业最后一部分，我们会暂时离开你亲手搭建的代码库，转向两个流行深度学习框架之一；这里使用 PyTorch。

## 为什么使用深度学习框架？

* 我们的代码现在可以在 GPU 上运行。这会让模型训练快得多。使用 PyTorch 这样的框架时，你可以为自己的神经网络架构利用 GPU 能力，而不必直接编写 CUDA 代码（这超出了本课程范围）。
* 在本课程中，我们希望你能为最终项目做好准备，使用这些框架比手写每个所需功能更高效。
* 我们希望你站在前人成果之上。PyTorch 是很优秀的框架，会让你的工作轻松很多；现在你已经理解了这些组件内部的大致机制，可以放心使用它们。
* 最后，我们希望你接触到学术界或工业界中常见的深度学习代码形式。

## 什么是 PyTorch？

PyTorch 是一个在 Tensor 对象上执行动态计算图的系统；Tensor 的行为类似 numpy ndarray。它带有强大的自动微分引擎，因此不需要手写反向传播。

## 如何学习 PyTorch？

我们以前的一位课程讲师 Justin Johnson 为 PyTorch 制作了优秀的 [tutorial](https://github.com/jcjohnson/pytorch-examples)。

你也可以在这里找到详细的 [API doc](http://pytorch.org/docs/stable/index.html)。如果你有 API 文档没有覆盖的问题，[PyTorch forum](https://discuss.pytorch.org/) 通常比 StackOverflow 更适合提问。

# 目录

本作业包含 5 个部分。你将从**三个不同抽象层次**学习 PyTorch，这有助于你更深入理解它，并为最终项目做准备。

1. Part I，准备：使用 CIFAR-10 数据集。
2. Part II，Barebones PyTorch：**抽象层次 1**，直接使用最底层的 PyTorch Tensor。
3. Part III，PyTorch Module API：**抽象层次 2**，使用 `nn.Module` 定义任意神经网络架构。
4. Part IV，PyTorch Sequential API：**抽象层次 3**，使用 `nn.Sequential` 非常方便地定义线性前馈网络。
5. Part V，CIFAR-10 开放挑战：实现你自己的网络，在 CIFAR-10 上尽可能取得高准确率。你可以尝试任意层、优化器、超参数或其他高级功能。

下面是一个对比表：

| API           | 灵活性 | 便利性 |
|---------------|--------|--------|
| Barebone      | 高     | 低     |
| `nn.Module`   | 高     | 中     |
| `nn.Sequential` | 低   | 高     |

# GPU

在 Colab 中，你可以点击 `Runtime -> Change runtime type`，并在 `Hardware Accelerator` 下选择 `GPU`，手动切换到 GPU 设备。请在运行下面导入包的单元之前完成这一步，因为切换 runtime 会重启内核。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as dset
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32 # 本教程全程使用 float。

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# 控制训练损失打印频率的常量。
print_every = 100
print('using device:', device)

# Part I：准备

现在加载 CIFAR-10 数据集。第一次执行可能需要几分钟，但文件之后应该会被缓存。

在本作业前面的部分中，我们需要自己写代码下载 CIFAR-10、预处理数据，并按 minibatch 迭代；PyTorch 提供了方便的工具，可以帮我们自动完成这些流程。

<!-- -->

In [ ]:
NUM_TRAIN = 49000

# torchvision.transforms 包提供数据预处理和数据增强工具；这里我们设置一个 transform，
# 通过减去 RGB 均值并除以各通道 RGB 标准差来预处理数据。均值和标准差已在此处硬编码。
transform = T.Compose([
                T.ToTensor(),
                T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
            ])

# 我们为每个划分（训练/验证/测试）创建一个 Dataset 对象；Dataset 会逐个加载训练样本，
# 因此再用 DataLoader 包装 Dataset，让它迭代 Dataset 并形成 minibatch。
# 通过向 DataLoader 传入 Sampler 对象，我们把 CIFAR-10 训练集划分为训练集和验证集，
# Sampler 会指定如何从底层 Dataset 中采样。
cifar10_train = dset.CIFAR10('./cs231n/datasets', train=True, download=True,
                             transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./cs231n/datasets', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('./cs231n/datasets', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

# Part II：Barebones PyTorch

PyTorch 自带高层 API，可以方便地定义模型架构；这些会在本教程 Part III 中介绍。在本节中，我们先从 barebone PyTorch 元素开始，以便更好理解 autograd 引擎。完成这个练习后，你会更理解高层模型 API 的价值。

我们将从一个简单的全连接 ReLU 网络开始，它包含两个隐藏层、没有 bias，用于 CIFAR 分类。这个实现使用 PyTorch Tensor 运算计算前向传播，并使用 PyTorch autograd 计算梯度。你需要理解每一行代码，因为后面你会写一个更难的版本。

当我们创建一个 `requires_grad=True` 的 PyTorch Tensor 时，涉及这个 Tensor 的运算不仅会计算数值，还会在后台构建计算图，使我们能够方便地对计算图做反向传播，计算某些 Tensor 关于下游损失的梯度。具体来说，如果 `x` 是一个 Tensor 且 `x.requires_grad == True`，那么反向传播后 `x.grad` 会是另一个 Tensor，保存最终标量损失对 `x` 的梯度。

### PyTorch Tensor：Flatten 函数
从概念上看，PyTorch Tensor 类似 numpy 数组：它是一个 n 维数字网格，并且像 numpy 一样提供许多高效操作 Tensor 的函数。作为简单例子，我们在下面提供一个 `flatten` 函数，它会把图像数据 reshape 成全连接神经网络可用的形式。

回忆一下，图像数据通常存储为形状 `N x C x H x W` 的 Tensor，其中：

* `N` 是数据点数量。
* `C` 是通道数。
* `H` 是中间特征图的像素高度。
* `W` 是中间特征图的像素宽度。

当我们做二维卷积等需要理解中间特征空间位置关系的操作时，这是表示数据的正确方式。但当我们使用全连接 affine 层处理图像时，希望每个数据点由一个单独向量表示；此时再区分数据的通道、行和列就不再有用。因此，我们使用 “flatten” 操作，把每个样本的 `C x H x W` 值折叠成一个长向量。下面的 flatten 函数先读取给定 batch 数据中的 `N`、`C`、`H` 和 `W`，然后返回这个数据的一个 “view”。“View” 类似 numpy 的 “reshape” 方法：它把 `x` 的维度改成 `N x ??`，其中 `??` 可以是任意值（这里是 `C x H x W`，但我们不需要显式指定）。

<!-- -->

In [ ]:
def flatten(x):
    N = x.shape[0] # 读入 N、C、H、W
    return x.view(N, -1)  # "flatten" C * H * W 值 到 a single vector per image

def test_flatten():
    x = torch.arange(12).view(2, 1, 3, 2)
    print('Before flattening: ', x)
    print('After flattening: ', flatten(x))

test_flatten()

### Barebones PyTorch：两层网络

这里定义 `two_layer_fc` 函数，它对一批图像数据执行两层全连接 ReLU 网络的前向传播。定义前向传播后，我们通过把全零输入传入网络来检查它不会崩溃，并且输出形状正确。

这里不需要你写代码，但阅读并理解实现很重要。

<!-- -->

In [ ]:
import torch.nn.functional as F  # 有用的无状态函数

def two_layer_fc(x, params):
    """
    一个全连接神经网络，结构为：
    fully connected -> ReLU -> fully connected。
    注意：这个函数只定义前向传播；PyTorch 会负责反向传播。

    网络输入是一个 minibatch，形状为 (N, d1, ..., dM)，其中 d1 * ... * dM = D。
    隐藏层包含 H 个单元，输出层会为 C 个类别产生分数。

    输入:
    - x: 形状为 (N, d1, ..., dM) 的 PyTorch Tensor，表示一个输入数据 minibatch。
    - params: PyTorch Tensor 列表 [w1, w2]，表示网络权重；
      w1 的形状为 (D, H)，w2 的形状为 (H, C)。

    返回:
    - scores: 形状为 (N, C) 的 PyTorch Tensor，给出输入数据 x 的分类分数。
    """
    # 首先 flatten 图像
    x = flatten(x)  # 形状: [batch_size, C x H x W]

    w1, w2 = params

    # 前向传播：使用 Tensor 操作计算预测值。由于 w1 和 w2 设置了 requires_grad=True，
    # 涉及这些 Tensor 的操作会让 PyTorch 构建计算图，从而自动计算梯度。
    # 现在不再手写反向传播，因此无需保存中间值引用。
    # 你也可以使用 `.clamp(min=0)`，它等价于 F.relu()。
    x = F.relu(x.mm(w1))
    x = x.mm(w2)
    return x


def two_layer_fc_test():
    hidden_layer_size = 42
    x = torch.zeros((64, 50), dtype=dtype)  # minibatch size 64, 特征维度 50
    w1 = torch.zeros((50, hidden_layer_size), dtype=dtype)
    w2 = torch.zeros((hidden_layer_size, 10), dtype=dtype)
    scores = two_layer_fc(x, [w1, w2])
    print(scores.size())  # 你应该看到 [64, 10]。

two_layer_fc_test()

### Barebones PyTorch：三层 ConvNet

这里你需要完成 `three_layer_convnet` 函数，它会执行三层卷积网络的前向传播。和上面类似，我们可以通过把全零输入传入网络，立即测试实现。网络架构如下：

1. 一个带 bias 的卷积层，包含 `channel_1` 个滤波器，每个滤波器形状为 `KW1 x KH1`，zero-padding 为 2。
2. ReLU 非线性。
3. 一个带 bias 的卷积层，包含 `channel_2` 个滤波器，每个滤波器形状为 `KW2 x KH2`，zero-padding 为 1。
4. ReLU 非线性。
5. 带 bias 的全连接层，输出 C 个类别的分数。

注意，全连接层之后这里**没有 softmax activation**：这是因为 PyTorch 的 cross entropy loss 会替你执行 softmax activation，并且把这一步合并进去可以提高计算效率。

**提示**：关于卷积请参考 http://pytorch.org/docs/stable/nn.html#torch.nn.functional.conv2d；注意卷积滤波器的形状。

<!-- -->

In [ ]:
def three_layer_convnet(x, params):
    """
    按照上面定义的结构，执行三层卷积网络的前向传播。

    输入:
    - x: 形状为 (N, 3, H, W) 的 PyTorch Tensor，表示一个图像 minibatch。
    - params: PyTorch Tensor 列表，给出网络的权重和偏置；
      应包含以下内容：
      - conv_w1: 形状为 (channel_1, 3, KH1, KW1) 的 PyTorch Tensor，表示第一层卷积权重。
      - conv_b1: 形状为 (channel_1,) 的 PyTorch Tensor，表示第一层卷积偏置。
      - conv_w2: 形状为 (channel_2, channel_1, KH2, KW2) 的 PyTorch Tensor，表示第二层卷积权重。
      - conv_b2: 形状为 (channel_2,) 的 PyTorch Tensor，表示第二层卷积偏置。
      - fc_w: PyTorch Tensor，表示全连接层权重。你能推断它的形状吗？
      - fc_b: PyTorch Tensor，表示全连接层偏置。你能推断它的形状吗？

    返回:
    - scores: 形状为 (N, C) 的 PyTorch Tensor，表示 x 的分类分数。
    """
    conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b = params
    scores = None
    ################################################################################
    # TODO：实现三层 ConvNet 的前向传播。                      #
    ################################################################################

    ################################################################################
    #                                 你的代码结束                             #
    ################################################################################
    return scores

定义好上面 ConvNet 的前向传播后，运行下面的单元测试你的实现。

运行这个函数时，`scores` 的形状应为 `(64, 10)`。

<!-- -->

In [ ]:
def three_layer_convnet_test():
    x = torch.zeros((64, 3, 32, 32), dtype=dtype)  # minibatch size 64, 图像大小 [3, 32, 32]

    conv_w1 = torch.zeros((6, 3, 5, 5), dtype=dtype)  # [out_channel, in_channel, kernel_H, kernel_W]
    conv_b1 = torch.zeros((6,))  # out_channel
    conv_w2 = torch.zeros((9, 6, 3, 3), dtype=dtype)  # [out_channel, in_channel, kernel_H, kernel_W]
    conv_b2 = torch.zeros((9,))  # out_channel

    # you must calculate 形状 的 tensor after two conv 层, before fully-connected 层
    fc_w = torch.zeros((9 * 32 * 32, 10))
    fc_b = torch.zeros(10)

    scores = three_layer_convnet(x, [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b])
    print(scores.size())  # 你应该看到 [64, 10]。
three_layer_convnet_test()

### Barebones PyTorch：初始化
我们来写几个工具方法，用于初始化模型的权重矩阵。

- `random_weight(shape)` 使用 Kaiming normalization 方法初始化权重 Tensor。
- `zero_weight(shape)` 初始化全零 Tensor，适合用于实例化 bias 参数。

`random_weight` 函数使用 Kaiming normal 初始化方法，见：

He et al, *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*, ICCV 2015, https://arxiv.org/abs/1502.01852

<!-- -->

In [ ]:
def random_weight(shape):
    """
    创建用于权重的随机 Tensor；设置 requires_grad=True 表示
    我们希望在反向传播期间为这些 Tensor 计算梯度。
    这里使用 Kaiming 归一化：sqrt(2 / fan_in)。
    """
    if len(shape) == 2:  # FC weight
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:]) # conv weight [out_channel, in_channel, kH, kW]
    # randn is standard normal distribution generator.
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

# 创建形状为 [3 x 5] 的权重。
# 如果使用 GPU，你应该看到类型为 `torch.cuda.FloatTensor`；
# 否则类型应为 `torch.FloatTensor`。
random_weight((3, 5))

### Barebones PyTorch：检查准确率
训练模型时，我们将使用下面的函数检查模型在训练集或验证集上的准确率。

检查准确率时不需要计算任何梯度，因此计算 scores 时也不需要 PyTorch 为我们构建计算图。为了阻止计算图被构建，我们把计算放在 `torch.no_grad()` context manager 中。

<!-- -->

In [ ]:
def check_accuracy_part2(loader, model_fn, params):
    """
    Check accuracy 的 a 分类 模型.

    输入:
    - loader: A DataLoader 用于 数据 split we want 到 check
    - 模型_fn: A 函数 该 performs 前向传播 模型的,
      使用 signature 分数 = 模型_fn(x, params)
    - params: List 的 PyTorch Tensors giving 参数 模型的

    返回: Nothing, but prints accuracy 模型的
    """
    split = 'val' if loader.dataset.train else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # 移动到设备，例如 GPU
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))

### Barebones PyTorch：训练循环
现在可以搭建一个基础训练循环来训练网络。我们将使用不带 momentum 的随机梯度下降训练模型，并用 `torch.functional.cross_entropy` 计算损失；你可以在[这里阅读相关文档](http://pytorch.org/docs/stable/nn.html#cross-entropy)。

训练循环接收神经网络函数、初始化后的参数列表（例如例子中的 `[w1, w2]`）和学习率。

<!-- -->

In [ ]:
def train_part2(model_fn, params, learning_rate):
    """
    在 CIFAR-10 上训练模型。

    输入:
    - model_fn: 执行模型前向传播的 Python 函数。它的签名应为
      scores = model_fn(x, params)，其中 x 是包含图像数据的 PyTorch Tensor，
      params 是给出模型权重的 PyTorch Tensor 列表，scores 是形状为 (N, C) 的
      PyTorch Tensor，表示 x 中各元素的分类分数。
    - params: 给出模型权重的 PyTorch Tensor 列表。
    - learning_rate: Python 标量，表示 SGD 使用的学习率。

    返回: None
    """
    for t, (x, y) in enumerate(loader_train):
        # 将数据移动到合适设备（GPU 或 CPU）
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)

        # 前向传播：计算分数和损失。
        scores = model_fn(x, params)
        loss = F.cross_entropy(scores, y)

        # 反向传播：PyTorch 会找出计算图中哪些 Tensor 设置了 requires_grad=True，
        # 并通过反向传播计算损失对这些 Tensor 的梯度，
        # 然后把梯度存入每个 Tensor 的 .grad 属性。
        loss.backward()

        # 更新参数。我们不希望对参数更新本身继续反向传播，
        # 因此把更新放在 torch.no_grad() 上下文管理器中，
        # 防止 PyTorch 为这一步构建计算图。
        with torch.no_grad():
            for w in params:
                w -= learning_rate * w.grad

                # 运行反向传播后手动清零梯度
                w.grad.zero_()

        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(loader_val, model_fn, params)
            print()

### Barebones PyTorch：训练两层网络
现在可以运行训练循环了。我们需要显式分配全连接权重 `w1` 和 `w2` 的 Tensor。

CIFAR 的每个 minibatch 有 64 个样本，因此 Tensor 形状为 `[64, 3, 32, 32]`。

flatten 后，`x` 的形状应为 `[64, 3 * 32 * 32]`。这会是 `w1` 第一维的大小。`w1` 的第二维是隐藏层大小，同时也是 `w2` 第一维的大小。

最后，网络输出是一个 10 维向量，表示 10 个类别上的概率分布。

你不需要调任何超参数，但训练一个 epoch 后应该看到高于 40% 的准确率。

<!-- -->

In [ ]:
hidden_layer_size = 4000
learning_rate = 1e-2

w1 = random_weight((3 * 32 * 32, hidden_layer_size))
w2 = random_weight((hidden_layer_size, 10))

train_part2(two_layer_fc, [w1, w2], learning_rate)

### Barebones PyTorch：训练 ConvNet

下面你应该使用上面定义的函数，在 CIFAR 上训练一个三层卷积网络。网络架构如下：

1. 带 bias 的卷积层，包含 32 个 `5x5` 滤波器，zero-padding 为 2。
2. ReLU。
3. 带 bias 的卷积层，包含 16 个 `3x3` 滤波器，zero-padding 为 1。
4. ReLU。
5. 带 bias 的全连接层，计算 10 个类别的分数。

你应该使用上面定义的 `random_weight` 函数初始化权重矩阵，并使用 `zero_weight` 函数初始化 bias 向量。

你不需要调任何超参数；如果一切正常，训练一个 epoch 后应达到 42% 以上的准确率。

<!-- 重要提示：为了保证你的解答能正确通过 autograder，请确保在解答中使用变量 `flat_feat_dim`。我们会用它确认你的维度是否正确，因此这一步非常重要。没有正确使用会导致得分为 0。 -->

In [ ]:
learning_rate = 3e-3

channel_1 = 32
channel_2 = 16

conv_w1 = None
conv_b1 = None
conv_w2 = None
conv_b2 = None
fc_w = None
fc_b = None

################################################################################
# TODO: 初始化 参数 的 a three-层 ConvNet.                    #
################################################################################

################################################################################
#                                 你的代码结束                             #
################################################################################

params = [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b]
train_part2(three_layer_convnet, params, learning_rate)

# Part III：PyTorch Module API

Barebone PyTorch 要求我们手动跟踪所有参数 Tensor。对于只有少数 Tensor 的小网络，这还可以接受；但对包含数十或数百个 Tensor 的大网络来说，这会非常不方便且容易出错。

PyTorch 提供了 `nn.Module` API，让你可以定义任意网络架构，同时自动跟踪所有可学习参数。在 Part II 中，我们自己实现了 SGD。PyTorch 还提供 `torch.optim` 包，实现了常见优化器，例如 RMSProp、Adagrad 和 Adam。它甚至支持 L-BFGS 这样的近似二阶方法。你可以参考 [doc](http://pytorch.org/docs/master/optim.html) 查看每个优化器的具体说明。

使用 Module API 时，请按以下步骤：

1. 继承 `nn.Module`。给你的网络类起一个直观名字，例如 `TwoLayerFC`。

2. 在构造函数 `__init__()` 中，把所需层定义为类属性。`nn.Linear` 和 `nn.Conv2d` 等层对象本身也是 `nn.Module` 的子类，并包含可学习参数，因此你不必自己实例化原始 Tensor。`nn.Module` 会为你跟踪这些内部参数。参考 [doc](http://pytorch.org/docs/master/nn.html) 了解更多内置层。**警告**：不要忘记先调用 `super().__init__()`。

3. 在 `forward()` 方法中定义网络的*连接关系*。你应该把 `__init__` 中定义的属性作为函数调用，接收 Tensor 输入并输出“变换后”的 Tensor。不要在 `forward()` 中创建任何带可学习参数的新层；它们都必须提前在 `__init__` 中声明。

定义好 Module 子类后，你可以实例化它，并像 Part II 中的神经网络前向函数一样调用它。

### Module API：两层网络
下面是一个两层全连接网络的具体例子：

<!-- -->

In [ ]:
class TwoLayerFC(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        # 将层对象赋给类属性
        self.fc1 = nn.Linear(input_size, hidden_size)
        # nn.init package contains convenient initialization methods
        # http://pytorch.org/docs/master/nn.html#torch-nn-init
        nn.init.kaiming_normal_(self.fc1.weight)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        nn.init.kaiming_normal_(self.fc2.weight)

    def forward(self, x):
        # 前向 始终定义连接关系
        x = flatten(x)
        scores = self.fc2(F.relu(self.fc1(x)))
        return scores

def test_TwoLayerFC():
    input_size = 50
    x = torch.zeros((64, input_size), dtype=dtype)  # minibatch size 64, 特征维度 50
    model = TwoLayerFC(input_size, 42, 10)
    scores = model(x)
    print(scores.size())  # 你应该看到 [64, 10]。
test_TwoLayerFC()

### Module API：三层 ConvNet
现在轮到你实现一个三层 ConvNet，并接一个全连接层。网络架构应与 Part II 相同：

1. 卷积层，包含 `channel_1` 个 `5x5` 滤波器，zero-padding 为 2。
2. ReLU。
3. 卷积层，包含 `channel_2` 个 `3x3` 滤波器，zero-padding 为 1。
4. ReLU。
5. 全连接层，输出 `num_classes` 个类别。

你应该使用 Kaiming normal 初始化方法初始化模型权重矩阵。

**提示**：http://pytorch.org/docs/stable/nn.html#conv2d

实现三层 ConvNet 后，`test_ThreeLayerConvNet` 函数会运行你的实现；它应该打印输出分数的形状 `(64, 10)`。

<!-- -->

In [ ]:
class ThreeLayerConvNet(nn.Module):
    def __init__(self, in_channel, channel_1, channel_2, num_classes):
        super().__init__()
        ########################################################################
        # TODO：设置 层 you 需要 用于 a three-层 ConvNet 使用  #
        # architecture defined above.                                          #
        ########################################################################

        ########################################################################
        #                          你的代码结束                            #
        ########################################################################

    def forward(self, x):
        scores = None
        ########################################################################
        # TODO：实现 前向 函数 用于 a 3-层 ConvNet. you      #
        # 应该 使用 层 you defined 在 __init__ 并 specify        #
        # connectivity 的 those 层 在 前向()                            #
        ########################################################################

        ########################################################################
        #                             你的代码结束                         #
        ########################################################################
        return scores


def test_ThreeLayerConvNet():
    x = torch.zeros((64, 3, 32, 32), dtype=dtype)  # minibatch size 64, 图像大小 [3, 32, 32]
    model = ThreeLayerConvNet(in_channel=3, channel_1=12, channel_2=8, num_classes=10)
    scores = model(x)
    print(scores.size())  # 你应该看到 [64, 10]。
test_ThreeLayerConvNet()

### Module API：检查准确率
给定验证集或测试集，我们可以检查神经网络的分类准确率。

这个版本和 Part II 中的版本略有不同。你不再需要手动传入参数。

<!-- -->

In [ ]:
def check_accuracy_part34(loader, model):
    if loader.dataset.train:
        print('Checking accuracy on validation set')
    else:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()  # 将模型设置为评估模式
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # 移动到设备，例如 GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))

### Module API：训练循环
这里也使用略有不同的训练循环。我们不再自己更新权重值，而是使用 `torch.optim` 包中的 Optimizer 对象；它抽象了优化算法这一概念，并提供了大多数常用神经网络优化算法的实现。

<!-- -->

In [ ]:
def train_part34(model, optimizer, epochs=1):
    """
    Train a 模型 on CIFAR-10 使用 PyTorch Module API.

    输入:
    - 模型: A PyTorch Module giving 模型 到 训练.
    - optimizer: An Optimizer object 我们将 使用 到 训练 模型
    - epochs: (Optional) A Python integer giving 数量 epochs 到 训练 for

    返回: Nothing, but prints 模型 accuracies during 训练.
    """
    model = model.to(device=device)  # 将模型参数移动到 CPU/GPU
    for e in range(epochs):
        for t, (x, y) in enumerate(loader_train):
            model.train()  # 将模型设置为训练模式
            x = x.to(device=device, dtype=dtype)  # 移动到设备，例如 GPU
            y = y.to(device=device, dtype=torch.long)

            scores = model(x)
            loss = F.cross_entropy(scores, y)

            # Zero out 所有 的 梯度 用于 变量 which optimizer
            # 将 update.
            optimizer.zero_grad()

            # This is 反向 pass: 计算 梯度 的 损失 使用
            # respect 到 each  parameter 模型的.
            loss.backward()

            # 使用梯度实际更新模型参数
            # 计算得到的 by 反向 pass.
            optimizer.step()

            if t % print_every == 0:
                print('Iteration %d, loss = %.4f' % (t, loss.item()))
                check_accuracy_part34(loader_val, model)
                print()

### Module API：训练两层网络
现在可以运行训练循环了。与 Part II 不同，我们不再显式分配参数 Tensor。

只需把输入大小、隐藏层大小和类别数（即输出大小）传给 `TwoLayerFC` 的构造函数。

你还需要定义一个 optimizer，用来跟踪 `TwoLayerFC` 内部所有可学习参数。

你不需要调任何超参数，但训练一个 epoch 后应该看到高于 40% 的模型准确率。

<!-- -->

In [ ]:
hidden_layer_size = 4000
learning_rate = 1e-2
model = TwoLayerFC(3 * 32 * 32, hidden_layer_size, 10)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

train_part34(model, optimizer)

### Module API：训练三层 ConvNet
现在你应该使用 Module API 在 CIFAR 上训练一个三层 ConvNet。这应该和训练两层网络非常相似。你不需要调任何超参数，但训练一个 epoch 后应该达到 45% 以上的准确率。

请使用不带 momentum 的随机梯度下降训练模型。

<!-- -->

In [ ]:
learning_rate = 3e-3
channel_1 = 32
channel_2 = 16

model = None
optimizer = None
################################################################################
# TODO：实例化你的 ThreeLayerConvNet 模型以及对应的 optimizer。 #
################################################################################

################################################################################
#                                 你的代码结束                             #
################################################################################

train_part34(model, optimizer)

# Part IV：PyTorch Sequential API

Part III 介绍了 PyTorch Module API，它允许你定义任意可学习层及其连接关系。

对于简单模型，例如一串前馈层，你仍然需要经历 3 步：继承 `nn.Module`，在 `__init__` 中把层赋给类属性，并在 `forward()` 中逐个调用每一层。有没有更方便的方法？

幸运的是，PyTorch 提供了一个容器 Module，叫 `nn.Sequential`，它把上述步骤合并成一步。它不如 `nn.Module` 灵活，因为你不能指定比前馈堆叠更复杂的拓扑，但对许多场景来说已经足够。

### Sequential API：两层网络
我们来看如何用 `nn.Sequential` 重写两层全连接网络示例，并使用上面定义的训练循环来训练它。

同样，这里不需要调任何超参数，但训练一个 epoch 后应该达到 40% 以上的准确率。

<!-- -->

In [ ]:
# 为了把 `flatten` 堆叠到 Sequential 中，需要将其包装为 module
# in nn.Sequential
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)

hidden_layer_size = 4000
learning_rate = 1e-2

model = nn.Sequential(
    Flatten(),
    nn.Linear(3 * 32 * 32, hidden_layer_size),
    nn.ReLU(),
    nn.Linear(hidden_layer_size, 10),
)

# 可以在 optim.SGD 中使用 Nesterov momentum
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)

train_part34(model, optimizer)

### Sequential API：三层 ConvNet
这里你应该使用 `nn.Sequential` 定义并训练一个三层 ConvNet，其架构与 Part III 中使用的相同：

1. 带 bias 的卷积层，包含 32 个 `5x5` 滤波器，zero-padding 为 2。
2. ReLU。
3. 带 bias 的卷积层，包含 16 个 `3x3` 滤波器，zero-padding 为 1。
4. ReLU。
5. 带 bias 的全连接层，计算 10 个类别的分数。

你可以使用 PyTorch 默认权重初始化。

请使用带 Nesterov momentum 0.9 的随机梯度下降优化模型。

同样，你不需要调任何超参数，但训练一个 epoch 后应该看到高于 55% 的准确率。

<!-- -->

In [ ]:
channel_1 = 32
channel_2 = 16
learning_rate = 1e-2

model = None
optimizer = None

################################################################################
# TODO：重写 2-层 ConvNet 使用 bias 来自 Part III 使用           #
# Sequential API.                                                              #
################################################################################

################################################################################
#                                 你的代码结束                             #
################################################################################

train_part34(model, optimizer)

# Part V：CIFAR-10 开放挑战

在这一节中，你可以在 CIFAR-10 上实验任何你喜欢的 ConvNet 架构。

现在你的任务是尝试不同架构、超参数、损失函数和优化器，训练一个在 10 个 epoch 内在 CIFAR-10 **验证集**上达到**至少 70%** 准确率的模型。你可以使用上面的 `check_accuracy` 和 `train` 函数，也可以使用 `nn.Module` 或 `nn.Sequential` API。

请在 notebook 末尾描述你做了什么。

下面是各组件的官方 API 文档。注意：课堂上称为 “spatial batch norm” 的东西，在 PyTorch 中叫 “BatchNorm2D”。

* torch.nn 包中的层：http://pytorch.org/docs/stable/nn.html
* 激活函数：http://pytorch.org/docs/stable/nn.html#non-linear-activations
* 损失函数：http://pytorch.org/docs/stable/nn.html#loss-functions
* 优化器：http://pytorch.org/docs/stable/optim.html

### 可以尝试的方向：
- **滤波器大小**：上面使用了 `5x5`；更小的滤波器会不会更高效？
- **滤波器数量**：上面使用了 32 个滤波器。更多或更少会不会更好？
- **Pooling vs Strided Convolution**：使用 max pooling，还是只使用 stride convolution？
- **Batch normalization**：尝试在卷积层后加入 spatial batch normalization，在 affine 层后加入普通 batch normalization。网络训练会更快吗？
- **网络架构**：上面的网络有两层可训练参数。使用更深网络能否做得更好？可以尝试的好架构包括：
    - `[conv-relu-pool]xN -> [affine]xM -> [softmax or SVM]`
    - `[conv-relu-conv-relu-pool]xN -> [affine]xM -> [softmax or SVM]`
    - `[batchnorm-relu-conv]xN -> [affine]xM -> [softmax or SVM]`
- **Global Average Pooling**：不要 flatten 后接多个 affine 层，而是持续卷积直到图像变小（例如 `7x7` 左右），然后执行 average pooling 得到 `1x1` 图像表示 `(1, 1, Filter#)`，再 reshape 成 `(Filter#)` 向量。这用在 [Google's Inception Network](https://arxiv.org/abs/1512.00567) 中（见其架构表 Table 1）。
- **正则化**：加入 L2 权重正则化，或使用 Dropout。

### 训练提示
对你尝试的每个网络架构，都应该调学习率和其他超参数。调参时请注意：

- 如果参数工作正常，几百次迭代内应该能看到提升。
- 记住超参数调优的 coarse-to-fine 方法：先用少量训练迭代测试较大范围的超参数，找到至少能工作的参数组合。
- 找到一些看起来有效的参数组合后，再围绕它们更精细地搜索。你可能需要训练更多 epoch。
- 应该使用验证集做超参数搜索，把测试集留到最后，用验证集选出的最佳参数评估你的架构。

### 进一步挑战
如果你想更进一步，还有很多功能可以实现来尝试提升性能。你**不需要**实现这些内容，但如果有时间也可以尝试。

- 替代优化器：Adam、Adagrad、RMSprop 等。
- 替代激活函数，例如 leaky ReLU、parametric ReLU、ELU 或 MaxOut。
- 模型集成。
- 数据增强。
- 新架构：
  - [ResNets](https://arxiv.org/abs/1512.03385)：把上一层输入加到输出上。
  - [DenseNets](https://arxiv.org/abs/1608.06993)：把前面层的输入拼接到一起。
  - [This blog has an in-depth overview](https://chatbotslife.com/resnets-highwaynets-and-densenets-oh-my-9bb15918ee32)

### 祝你训练顺利。

<!-- -->

In [ ]:
################################################################################
# TODO:                                                                        #
# 尝试任意网络结构、optimizer 和超参数。                                #
# 在 10 个 epoch 内，让验证集准确率至少达到 70%。                         #
#                                                                              #
# 注意：你可以通过把 loader_test 或 loader_val 作为第二个参数传给         #
# check_accuracy，在测试集或验证集上评估模型。在完成网络结构和超参数调优前， #
# 不要使用测试集；最后只运行一次测试集，用来报告最终结果。                 #
################################################################################
model = None
optimizer = None


################################################################################
#                                 你的代码结束                             #
################################################################################

# 你应该至少得到 70% 的准确率。
train_part34(model, optimizer, epochs=10)

## 描述你做了什么

在下面的单元中，写下你做了哪些尝试、实现了哪些额外功能，以及训练和评估网络过程中画了哪些图。

**回答：**

## 测试集：只运行一次

现在我们已经得到了满意的结果，可以在测试集上测试最终模型（你应该把它存储在 `best_model` 中）。思考一下它与验证集准确率相比如何。

<!-- -->

In [ ]:
best_model = model
check_accuracy_part34(loader_test, best_model)